# Pure Rule-Based CTG Classification (Relaxed Clinical Guidelines)

**3-class output**: Normal (0) | Suspect (1) | Pathologic (2)

Also saves a binary column: Normal (0) vs Abnormal (1).

## Pipeline
1. Read raw WFDB CTG signals (FHR + UC)
2. Downsample to 1 Hz, take last 60 min
3. Extract aligned features
4. Apply **Clinically Relaxed** Rule Engine (Allows mild decels and extremely 1Hz compressed variability)
5. Save `patient_labels.csv`

In [ ]:
!pip install wfdb -q

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import find_peaks
warnings.filterwarnings('ignore')

## Paths — Update before running

In [ ]:
DATA_DIR   = '/kaggle/input/datasets/annapoornaak/phase2-dataset/ctu-chb-intrapartum-cardiotocography-database-1.0.0'
OUTPUT_DIR = '/kaggle/working/processed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✓ Output directory ready: {OUTPUT_DIR}')

In [ ]:
hea_files    = glob.glob(os.path.join(DATA_DIR, '*.hea'))
record_names = sorted([os.path.basename(f).replace('.hea', '') for f in hea_files])
print(f'✓ Found {len(record_names)} records')

## Signal Processing Functions

In [ ]:
def prepare_signal(record_name):
    """
    Prepares the FHR and UC signals according to the DeepCTG paper pipeline:
    1. 1Hz Downsampling by averaging.
    2. Linear interpolation for missing gaps < 10 mins (600 seconds).
    3. Finds the latest valid continuous segment >= 10 mins.
    4. Extracts exactly 60 minutes (3600s), padding with NaNs if necessary.
    """
    record = wfdb.rdrecord(os.path.join(DATA_DIR, record_name))
    fhr = record.p_signal[:, 0].astype(float)
    uc  = record.p_signal[:, 1].astype(float)
    fs  = int(record.fs)
    fhr[fhr == 0] = np.nan
    uc[uc   == 0] = np.nan
    
    n = (len(fhr) // fs) * fs
    if n == 0:
        return None, None
        
    fhr_1hz = np.nanmean(fhr[:n].reshape(-1, fs), axis=1)
    uc_1hz  = np.nanmean(uc[:n].reshape(-1,  fs), axis=1)
    
    # 1. Linear interpolation for gaps < 10 minutes (600 seconds)
    def interpolate_small_gaps(sig, max_gap=600):
        s = pd.Series(sig)
        mask = s.isna()
        groups = mask.ne(mask.shift()).cumsum()
        gap_sizes = mask.groupby(groups).transform('size')
        large_gaps_mask = mask & (gap_sizes > max_gap)
        
        s_interp = s.interpolate(method='linear', limit_direction='both')
        s_interp[large_gaps_mask] = np.nan
        return s_interp.values

    fhr_1hz = interpolate_small_gaps(fhr_1hz)
    uc_1hz  = interpolate_small_gaps(uc_1hz)
    
    # 2. Extract the latest valid segment >= 10 mins without missing data
    is_valid = ~np.isnan(fhr_1hz)
    edges = np.diff(np.concatenate(([0], is_valid.view(np.int8), [0])))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0]
    
    valid_segments = [(s, e) for s, e in zip(starts, ends) if (e - s) >= 600]
    
    if not valid_segments:
        return None, None
        
    latest_s, latest_e = valid_segments[-1]
    
    # Exclude if the segment starts more than 90 mins (5400 sec) before recording end
    if len(fhr_1hz) - latest_s > 5400:
        return None, None
        
    fhr_seg = fhr_1hz[latest_s:latest_e]
    uc_seg  = uc_1hz[latest_s:latest_e]
    
    # 3. Target exactly 60 mins (3600 sec)
    if len(fhr_seg) > 3600:
        fhr_seg = fhr_seg[-3600:]
        uc_seg  = uc_seg[-3600:]
    elif len(fhr_seg) < 3600:
        pad_len = 3600 - len(fhr_seg)
        # Pad with NaNs to prevent breaking rule-based thresholds
        fhr_seg = np.pad(fhr_seg, (pad_len, 0), 'constant', constant_values=np.nan)
        # Assuming we pad with zeros for UC to signify no contractions, or NaNs? 
        # For rule engines NaNs are always physically safer
        uc_seg  = np.pad(uc_seg, (pad_len, 0), 'constant', constant_values=np.nan)
        
    return fhr_seg, uc_seg



def estimate_baseline(fhr):
    baseline = np.full_like(fhr, np.nan)
    for i in range(len(fhr)):
        s = max(0, i - 300)
        e = min(len(fhr), i + 300)
        seg = fhr[s:e]
        seg = seg[~np.isnan(seg)]
        if len(seg) > 0:
            baseline[i] = np.median(seg)
    return baseline

def count_events(mask, min_dur):
    events, dur = 0, 0
    for v in mask:
        if v:
            dur += 1
        else:
            if dur >= min_dur:
                events += 1
            dur = 0
    if dur >= min_dur:  # Handle event at the very end
        events += 1
    return events

## Feature Extraction

In [ ]:
def extract_features(record_name):
    fhr, uc = prepare_signal(record_name)
    if fhr is None:
        return None

    baseline = estimate_baseline(fhr)
    diff = fhr - baseline

    LB = float(np.nanmean(baseline))

    # amplitude-based variability (best for 1Hz)
    variability = np.nanmean(np.abs(diff))
    MSTV = float(variability)

    # ACCELERATIONS
    AC = count_events(diff >= 15, 15)

    # DECELERATIONS (CORRECTED)
    DL, DS, DP = 0, 0, 0
    mask = diff <= -15
    in_event = False
    start = 0

    for i, val in enumerate(mask):
        if val and not in_event:
            in_event = True
            start = i
        elif not val and in_event:
            end = i
            dur = end - start
            if dur >= 15:
                depth = np.nanmin(diff[start:end])
                if dur >= 120:
                    DP += 1
                elif depth <= -45:
                    DS += 1
                else:
                    DL += 1
            in_event = False
            
    # Handle event finishing at end of trace
    if in_event:
        end = len(mask)
        dur = end - start
        if dur >= 15:
            depth = np.nanmin(diff[start:end])
            if dur >= 120:
                DP += 1
            elif depth <= -45:
                DS += 1
            else:
                DL += 1

    # UC FEATURES
    # Replace NaNs with 0 for peak detection
    uc_clean = np.where(np.isnan(uc), 0, uc)
    peaks, _ = find_peaks(uc_clean, distance=60)
    UC = len(peaks)
    
    duration_min = len(uc_clean) / 60
    UC_rate = float((UC / duration_min) * 10)

    return dict(
        Patient_ID=record_name,
        LB=LB,
        MSTV=MSTV,
        AC=AC,
        DL=DL,
        DS=DS,
        DP=DP,
        UC=UC,
        UC_rate=UC_rate
    )

## Extract Features for All Records

In [ ]:
all_features, skipped = [], []
for i, rec in enumerate(record_names):
    try:
        feats = extract_features(rec)
        if feats is None: skipped.append(rec); continue
        all_features.append(feats)
        if (i + 1) % 50 == 0: print(f'  Processed {i+1}/{len(record_names)}')
    except Exception as e:
        print(f'  Failed {rec}: {e}'); skipped.append(rec)

df = pd.DataFrame(all_features)
print(f'\n✓ Features: {df.shape}  |  Skipped: {len(skipped)}')

## 🔍 Diagnostic — Check Feature Distributions
Run before classifying to understand your data and verify thresholds.

In [ ]:
print('=== Continuous Features ===')
print(df[['LB','MSTV','UC_rate']].describe().round(2))
print('\n=== Event Counts ===')
print(df[['AC','DL','DS','DP','UC']].describe().round(2))
print('\n=== Threshold Coverage ===')
print(f'  LB 110-160   (base normal) : {df.LB.between(110,160).sum()}')
print(f'  MSTV >= 2    (var  normal) : {(df.MSTV >= 2).sum()}')
print(f'  MSTV 1 - 2   (var  susp.)  : {df.MSTV.between(1, 2, inclusive="left").sum()}')
print(f'  MSTV < 1     (var  pathol.): {(df.MSTV < 1).sum()}')
print(f'  DS >= 5      (dec  pathol.): {(df.DS >= 5).sum()}')
print(f'  DP > 0       (dec  pathol.): {(df.DP > 0).sum()}')
print(f'  Any decel    (DL/DS/DP > 0): {((df.DL>0)|(df.DS>0)|(df.DP>0)).sum()}')
print(f'  AC == 0      (no accels)   : {(df.AC == 0).sum()}')

## NaN Imputation (column mean)

In [ ]:
for col in ['LB','MSTV','UC_rate']:
    m = df[col].mean(); n = df[col].isna().sum()
    if n: print(f'  Imputing {n} NaN in {col} with mean={m:.3f}')
    df[col] = df[col].fillna(m)

for col in ['AC','DL','DS','DP','UC']:
    m = df[col].mean(); n = df[col].isna().sum()
    if n: print(f'  Imputing {n} NaN in {col} with rounded mean={round(m)}')
    df[col] = df[col].fillna(round(m)).astype(int)

print(f'✓ Imputation done. Remaining NaNs: {df.isna().sum().sum()}')

## FIGO-Aligned Rule Engine (Adapted for 1Hz)

### Design principles:
| Component | Status |
|---|---|
| Baseline | ✅ unchanged |
| Accelerations | ✅ unchanged |
| Decelerations | ✅ mostly valid |
| Variability | ❌ adapted for 1Hz (smoothed signal → lower variability thresholds) |

**Final Statement:**
👉 "FIGO guidelines were adapted for 1 Hz downsampled CTG signals by modifying variability thresholds while preserving clinical interpretation principles."

In [ ]:
def figo_classify(row):
    LB   = row['LB']
    MSTV = row['MSTV']
    AC   = row['AC']
    DL   = row['DL']
    DS   = row['DS']
    DP   = row['DP']
    UC_rate = row['UC_rate']

    # BASELINE
    if 110 <= LB <= 160:
        base = 'normal'
    elif 100 <= LB < 110 or 160 < LB <= 180:
        base = 'suspicious'
    else:
        base = 'pathological'

    # VARIABILITY
    if MSTV < 1:
        var = 'pathological'
    elif MSTV < 2:
        var = 'suspicious'
    else:
        var = 'normal'

    # DECELERATIONS
    if DP > 0 or DS >= 5:
        dec = 'pathological'
    elif DS >= 2 or DL >= 2:
        dec = 'suspicious'
    else:
        dec = 'normal'

    # UC EFFECT (FIGO-like)
    if UC_rate > 7 and dec == 'suspicious':
        dec = 'pathological'

    # PATHOLOGICAL (FIXED)
    if (
        dec == 'pathological' or
        (var == 'pathological' and dec != 'normal')
    ):
        return 2

    # NORMAL
    if base == 'normal' and var != 'pathological' and dec == 'normal':
        return 0

    # SUSPECT
    return 1

## Apply Rules → Generate Labels

In [ ]:
label_map        = {0: 'Normal', 1: 'Suspect', 2: 'Pathologic'}
df['label']      = df.apply(figo_classify, axis=1)
df['label_name'] = df['label'].map(label_map)

# Also add binary column (Normal=0, Abnormal=1)
df['binary_label']      = df['label'].apply(lambda x: 0 if x == 0 else 1)
df['binary_label_name'] = df['binary_label'].map({0: 'Normal', 1: 'Abnormal'})

print('✓ Labels generated')
print('\n3-Class distribution:')
print(df['label_name'].value_counts())
print('\nBinary distribution:')
print(df['binary_label_name'].value_counts())
print()
df[['Patient_ID','LB','MSTV','AC','DL','DS','DP','UC','UC_rate','label','label_name']].head(10)

## Save Outputs

In [ ]:
# 3-class labels
df[['Patient_ID','label','label_name']].to_csv(
    os.path.join(OUTPUT_DIR, 'patient_labels.csv'), index=False)
print('✓ patient_labels.csv (3-class) saved')

# Binary labels
df[['Patient_ID','binary_label','binary_label_name']].rename(
    columns={'binary_label':'label','binary_label_name':'label_name'}
).to_csv(os.path.join(OUTPUT_DIR, 'patient_labels_binary.csv'), index=False)
print('✓ patient_labels_binary.csv saved')

# Full feature table
df.to_csv(os.path.join(OUTPUT_DIR, 'patient_features_and_labels.csv'), index=False)
print('✓ patient_features_and_labels.csv saved')

## Distribution Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 3-class
c3 = df['label_name'].value_counts()
color_map3 = {'Normal':'#2ecc71','Suspect':'#f39c12','Pathologic':'#e74c3c'}
colors3 = [color_map3.get(c,'#95a5a6') for c in c3.index]
bars = axes[0].bar(c3.index, c3.values, color=colors3)
for bar, val in zip(bars, c3.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('FIGO-Aligned Distribution', fontsize=12)
axes[0].set_ylabel('Records')

# Binary
c2 = df['binary_label_name'].value_counts()
color_map2 = {'Normal':'#2ecc71','Abnormal':'#e74c3c'}
colors2 = [color_map2.get(c,'#95a5a6') for c in c2.index]
bars = axes[1].bar(c2.index, c2.values, color=colors2, width=0.4)
for bar, val in zip(bars, c2.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 str(val), ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Binary Distribution', fontsize=12)
axes[1].set_ylabel('Records')

plt.suptitle('FIGO-Aligned Rule-Based Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'label_distribution.png'), dpi=120)
plt.show()
print('✓ Plot saved')